# 05 — Information Parity (IP)

**Notebook:** `05_information_parity.ipynb`  
**Part of:** `notebooks/indic/` pipeline  

This notebook computes **Information Parity (IP)** — a measure of how much more a language model struggles to compress Indic-script text compared to English. While Tokenization Parity (TP) captures surface-level fragmentation, IP captures the semantic dimension: does the model actually understand Indic languages as well as English?

We use BLOOM-560M's negative log-likelihood (NLL) as a proxy for compressibility. A lower NLL means the model finds the text more predictable (better understood). IP is defined as:

$$\text{IP} = \frac{\text{NLL}_{\text{total}}(\text{English source})}{\text{NLL}_{\text{total}}(\text{Indic sentence})}$$

| IP value | Interpretation |
|----------|---------------|
| 1.0 | Model compresses Indic as well as English — perfectly fair |
| 0.7 | Model finds Indic 30% harder than English — common for non-Latin scripts |
| 0.5 | Model finds Indic much harder than English — high representational bias |

> **Important:** We use **total NLL** (sum over tokens), not mean per-token NLL. Mean NLL would conflate IP with TP, since Indic sentences produce more tokens per word. Using the sum keeps the two metrics conceptually independent.

---
**Prerequisite:** Run `04_tokenization_parity.ipynb` first — this notebook reads the tokenized CSVs it produces.

## Setup

Install dependencies if needed:

```bash
pip install transformers torch accelerate pandas openpyxl
```

**BLOOM-560M** (BigScience, 2022) is a multilingual decoder-only language model trained on 45 natural languages including Hindi, but not the other four Indic languages in this study. This uneven coverage is itself a finding — it means IP is not only measuring script difficulty but also training data sparsity.

The first run will download the model weights (~1.1 GB). Subsequent runs use the Hugging Face cache.

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "bigscience/bloom-560m"
device     = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device : {device}")
print(f"Loading model: {MODEL_NAME}")
print("(First run will download ~1.1 GB — one-time download)\n")

ip_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# BLOOM defaults to left padding, which corrupts position embeddings in batched
# inference and inflates NLL for sentences at non-zero positions. We override
# this with right padding so real tokens always start at position 0.
ip_tokenizer.padding_side = "right"

ip_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
ip_model = ip_model.to(device)
ip_model.eval()

print("Model loaded.\n")

## The NLL Helper Function

The `compute_nll` function takes a list of text strings and returns the **total (summed) cross-entropy loss** for each, using a causal language modelling objective.

Key design choices:
- **Batched inference with right-padding** — attention masks exclude padding positions from the loss sum.
- **Total NLL, not mean** — mean NLL would fold TP into IP, destroying the independence of the two metrics.
- **NaN guards** — zero NLL is degenerate (perfect prediction), so any zero value is replaced with NaN before computing the ratio.

In [ ]:
import os
import glob
import pandas as pd

TOKENIZED_DIR = "../data/processed/tokenization_outputs"
ENGLISH_COL   = "Source"
BATCH_SIZE    = 8
MAX_LENGTH    = 256  # raised from 128 to reduce truncation on longer Indic sentences

# four columns we want IP for: native-script ref/trans and romanized ref/trans
IP_PAIRS = [
    ("Reference_xlmr_IP",                         "Reference"),
    ("Translation_xlmr_IP",                       "Translation"),
    ("Reference_Transliteration_romanized_xlmr_IP",   "Reference_Transliteration_romanized"),
    ("Translation_Transliteration_romanized_xlmr_IP", "Translation_Transliteration_romanized"),
]

In [ ]:
def compute_nll(texts, batch_size=BATCH_SIZE):
    """Return per-sentence total NLL for each string in `texts`.

    Uses total cross-entropy (sum over real tokens) rather than mean,
    so that IP and TP remain conceptually independent.
    """
    results = []

    for i in range(0, len(texts), batch_size):
        batch_raw = texts[i : i + batch_size]

        valid_indices, valid_texts = [], []
        for j, text in enumerate(batch_raw):
            if text and not (isinstance(text, float) and pd.isna(text)):
                valid_indices.append(j)
                valid_texts.append(str(text))

        batch_results = [float("nan")] * len(batch_raw)
        if not valid_texts:
            results.extend(batch_results)
            continue

        enc = ip_tokenizer(
            valid_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
        ).to(device)

        with torch.no_grad():
            logits = ip_model(**enc).logits

        for k, orig_idx in enumerate(valid_indices):
            input_ids  = enc["input_ids"][k]
            attn_mask  = enc["attention_mask"][k]

            # causal LM: position t predicts t+1, so shift by one
            shift_logits = logits[k, :-1, :]
            shift_labels = input_ids[1:]
            shift_mask   = attn_mask[1:].bool()

            if shift_mask.sum() == 0:
                batch_results[orig_idx] = float("nan")
                continue

            total_nll = F.cross_entropy(
                shift_logits[shift_mask],
                shift_labels[shift_mask],
                reduction="sum",
            )
            batch_results[orig_idx] = total_nll.item()

        results.extend(batch_results)

    return results

## Computing IP Across All Five Languages

For each language file we:
1. Compute total NLL for the English source (denominator, shared across all four IP columns)
2. Compute total NLL for each Indic column (numerator)
3. Divide to get IP and insert the result immediately after the text column it belongs to
4. Save back to the same CSV in place

This cell is safe to re-run — it checks whether the IP column already exists before inserting.

**IP pairs computed (native + romanized):**
- `Reference_xlmr_IP` — native-script reference
- `Translation_xlmr_IP` — native-script MT output
- `Reference_Transliteration_romanized_xlmr_IP` — romanized reference
- `Translation_Transliteration_romanized_xlmr_IP` — romanized MT output

In [ ]:
files = sorted(glob.glob(os.path.join(TOKENIZED_DIR, "*_xlmr_tokenized.csv")))
print(f"Found {len(files)} file(s)\n")

for path in files:
    fname = os.path.basename(path)
    df    = pd.read_csv(path)

    lang = next(
        (w.capitalize() for w in ["gujarati", "hindi", "tamil", "malayalam", "marathi"]
         if w in fname.lower()), "Unknown"
    )
    print(f"Processing: {lang} ...")

    # compute English NLL once — it's the denominator for every IP column
    nll_english = (
        pd.Series(compute_nll(df[ENGLISH_COL].fillna("").tolist()))
        .replace(0, float("nan"))  # zero NLL is degenerate
    )

    if "Source_NLL" not in df.columns:
        insert_at = df.columns.get_loc(ENGLISH_COL) + 1
        df.insert(insert_at, "Source_NLL", nll_english.values)

    for ip_col, text_col in IP_PAIRS:
        if text_col not in df.columns:
            print(f"  '{text_col}' not found in {fname} — skipped")
            continue
        if ip_col in df.columns:
            continue  # already computed

        nll_indic = (
            pd.Series(compute_nll(df[text_col].fillna("").tolist()))
            .replace(0, float("nan"))
        )

        ip_series = nll_english / nll_indic
        insert_at = df.columns.get_loc(text_col) + 1
        df.insert(insert_at, ip_col, ip_series.values)

    df.to_csv(path, index=False)

    col_summary = " ".join(
        f"{ip_col.split('_xlmr')[0].split('_Transliteration')[0].replace('_', ' ')} "
        f"IP={df[ip_col].mean():.3f}"
        for ip_col, _ in IP_PAIRS if ip_col in df.columns
    )
    print(f"✓ {lang:<12} | {col_summary}")

print("\nDone — IP columns added and files saved.")
print()
print("IP interpretation:")
print("  1.0 = model compresses Indic as well as English (perfectly fair)")
print("  0.7 = model finds Indic 30% harder than English (common for non-Latin scripts)")
print("  0.5 = model finds Indic much harder than English (high representational bias)")
print()
print("Compare with TP results:")
print("  TP = surface bias  (tokenizer splits the language more)")
print("  IP = semantic bias (model understands/compresses the language less)")

## Exporting to Excel

Combines all five CSV files (now containing both TP and IP columns) into a single Excel workbook for downstream analysis and visualization.

In [ ]:
excel_path = os.path.join(TOKENIZED_DIR, "Information_parity_outputs_all.xlsx")
csv_files  = sorted(glob.glob(os.path.join(TOKENIZED_DIR, "*.csv")))

if not csv_files:
    print(f"No CSV files found in: {TOKENIZED_DIR}")
else:
    with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
        for csv_path in csv_files:
            sheet_name = os.path.splitext(os.path.basename(csv_path))[0][:31]
            pd.read_csv(csv_path).to_excel(writer, sheet_name=sheet_name, index=False)
    print(f"✓ Combined Excel saved to: {excel_path}")

## References

1. **Tokenization bias in multilingual models:** Kanjirangat, V., Samardžić, T., Dolamic, L., & Rinaldi, F. (2025). Tokenization and Representation Biases in Multilingual Models on Dialectal NLP Tasks. *EMNLP 2025*, pp. 23992–24010. https://arxiv.org/abs/2509.20045

2. **Cross-lingual tokenization fairness:** Foroutan, N., Meister, C., Paul, D., Niklaus, J., Ahmadi, S., Bosselut, A., & Sennrich, R. (2025). Parity-Aware Byte-Pair Encoding: Improving Cross-lingual Fairness in Tokenization. arXiv:2508.04796. https://arxiv.org/abs/2508.04796

3. **BLOOM-560M (language model used for NLL):** BigScience Workshop. (2022). BLOOM: A 176B-Parameter Open-Access Multilingual Language Model. https://arxiv.org/abs/2211.05100

4. **XLM-RoBERTa (shared tokenizer backbone):** Conneau, A., Khandelwal, K., Goyal, N., Chaudhary, V., Wenzek, G., Guzmán, F., Grave, E., Ott, M., Zettlemoyer, L., & Stoyanov, V. (2020). Unsupervised Cross-lingual Representation Learning at Scale. *ACL 2020*. https://arxiv.org/abs/1911.02116

5. **IndicMT Eval dataset:** Sai B., A., Dixit, T., Nagarajan, V., Kunchukuttan, A., Kumar, P., Khapra, M. M., & Dabre, R. (2023). IndicMT Eval: A Dataset to Meta-Evaluate Machine Translation Metrics for Indian Languages. *ACL 2023*. https://aclanthology.org/2023.acl-long.795